[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb)

# 🟢 Easy: Gradient Norm Clipping

Implement **gradient norm clipping** — a training stability technique.

### Signature
```python
def clip_grad_norm(parameters, max_norm: float) -> float:
    # Clip gradients in-place so total norm <= max_norm
    # Returns the original (unclipped) total norm
```

### Algorithm
1. Compute total norm: `sqrt(sum(p.grad.norm()^2 for p in parameters))`
2. If total > max_norm: scale all grads by `max_norm / total`
3. Return original total norm

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [87]:
# ✏️ YOUR IMPLEMENTATION HERE
import math

def clip_grad_norm(parameters, max_norm):
  print(parameters)
  # 先计算所有parameters的L2-Norm
  # 这题的iput是[tensor([...]), tensor([...]), ...]，所以para.grad.norm()会为每个tensor计算L2-Norm，最后我们自己按照L2-Norm公式求平方和
  grads = [para.grad.norm() for para in parameters if para.grad is not None]
  total_norm = torch.sqrt(sum(grad ** 2 for grad in grads))
  # 如果L2-Norm比threshold大，那么每一个para的grad会乘上ratio
  if total_norm.item() > max_norm:
    ratio = max_norm / (total_norm.item() + 1e-6)
    for para in parameters:
      para.grad *= ratio
  return total_norm.item()

In [57]:
# 🧪 Debug
p = torch.randn(100, requires_grad=True)
(p * 10).sum().backward()
print('Before:', p.grad.norm().item())
orig = clip_grad_norm([p], max_norm=1.0)
print('After: ', p.grad.norm().item())
print('Original norm:', orig)

Before: 100.0
100.0
After:  0.9999999403953552
Original norm: 100.0


In [88]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_clipping')


🧪 Testing: Gradient Norm Clipping (Easy)
──────────────────────────────────────────────────
[tensor([ 1.3232,  0.0371, -0.2849, -0.1334,  1.8929,  3.1110, -0.4584, -0.3360,
        -1.5700,  1.2315], requires_grad=True), tensor([ 1.3946,  1.1711,  0.4335, -1.7343, -1.3360,  0.8871,  0.7680,  0.0571,
         0.2240,  0.5520], requires_grad=True)]
  ✅ [1/4] Clips to max_norm (4.1ms)
[tensor([-0.5788,  0.0177,  0.1318,  1.0198, -0.4468,  0.4520, -0.9759,  0.7112,
        -0.7582, -0.6436], requires_grad=True)]
  ✅ [2/4] Returns original norm (1.2ms)
[tensor([-0.6462, -0.1591, -1.7787,  0.8477], requires_grad=True)]
  ✅ [3/4] No change when norm < max_norm (0.8ms)
[tensor([-1.1258e+00, -1.1524e+00, -2.5058e-01, -4.3388e-01,  8.4871e-01,
         6.9201e-01, -3.1601e-01, -2.1152e+00,  3.2227e-01, -1.2633e+00,
         3.4998e-01,  3.0813e-01,  1.1984e-01,  1.2377e+00,  1.1168e+00,
        -2.4728e-01, -1.3527e+00, -1.6959e+00,  5.6665e-01,  7.9351e-01,
         5.9884e-01, -1.5551e+00, -3